# IndexTTS 2.5

- **支持显卡**：A100 (40GB / 80GB)、H100 (80GB)、RTX 6000 (96GB)
- **默认配置**：IndexTTS-2.5 BF16（GPT + Zipformer S2Mel + BigVGAN + QwenEmotion）、支持中 / 英 / 日 / 西 / 阿 5 语种零样本克隆、情绪控制与语速调节
- **访问方式**：默认使用 Cloudflare Quick Tunnel 临时公网域名（用完即关）；勾选 `USE_CF_TUNNEL_TOKEN` 则从 Colab Secrets 读取 `CF_TUNNEL_TOKEN` 挂载自有域名
- **执行顺序**：`1. 环境检测` → `2. 安装 ComfyUI` → `3. 下载模型与工作流` → `4. 启动并打开 ComfyUI`

| 工作流 | 功能 | 输入要求 |
| --- | --- | --- |
| `01_IndexTTS_2_5_Clone` | 基础声音克隆 | 1 段音色参考音频 + 文本（可选填写 `emo_text` 文字情绪描述） |
| `02_IndexTTS_2_5_Emotion_Vector` | 情绪滑块控制克隆 | 1 段音色参考音频 + 8 维情绪滑块（`0~1`）+ 文本 |
| `03_IndexTTS_2_5_Emotion_Ref` | 音色与情绪分离克隆 | 音色参考音频 + 情绪参考音频 + 文本 |

### 1. 环境检测
- **功能**：检测 GPU 型号、显存容量、BF16 支持、系统内存、磁盘空间及 PyTorch / CUDA 版本。
- **要求**：支持 `A100`、`H100`、`RTX 6000`，显存 40GB 及以上。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 1. 环境检测 {display-mode: "form"}
import os, shutil, subprocess, torch

MIN_VRAM_GIB = 35
REQUIREMENT = "40GB 及以上显存（A100 40GB / 80GB、H100 80GB、RTX 6000 96GB）"

if not torch.cuda.is_available():
    raise RuntimeError("未检测到 GPU。请在 Colab 菜单选择：代码执行程序 -> 更改运行时类型 -> A100 / H100 / RTX 6000 GPU。")

print(subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout)

gpu = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_gib = shutil.disk_usage("/content").free / 1024**3
bf16 = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu} | 显存: {vram_gib:.1f} GiB | 系统内存: {ram_gib:.1f} GiB | 磁盘可用: {disk_gib:.1f} GiB | BF16: {bf16}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

if not any(k in gpu.upper() for k in ("A100", "H100", "6000")) or vram_gib < MIN_VRAM_GIB:
    raise RuntimeError(f"当前显卡为 {gpu} ({vram_gib:.1f} GiB)，本笔记本需要 {REQUIREMENT}，请切换 runtime。")
if not bf16:
    raise RuntimeError("当前 GPU 不支持 BF16，请切换 runtime。")
print(f"[OK] GPU 环境正常：{gpu} ({vram_gib:.1f} GiB)")

### 2. 安装 ComfyUI
- **功能**：安装 ComfyUI `v0.43.2`、`ComfyUI-IndexTTS-2.5` 节点、依赖与 `cloudflared` `2026.10.6`。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 2. 安装 ComfyUI {display-mode: "form"}
import re, subprocess, sys, urllib.request
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
COMFY_REPO = "https://github.com/Comfy-Org/ComfyUI"
COMFY_VERSION = "v0.43.2"
COMFY_REVISION = "423cbd20511becb1f16122ab518e92e58bfa03d9"
CLOUDFLARED_VERSION = "2026.10.6"
NODE_DIR = COMFY_DIR / "custom_nodes/ComfyUI-IndexTTS-2_5"
NODE_REPO = "https://github.com/mailzwj/ComfyUI-IndexTTS-2.5"
NODE_REVISION = "a501798540c8079530b0e03d824c79763d12e7bd"
EXTRA_PACKAGES = ["transformers==4.57.6", "wetext>=0.0.9", "fugashi>=1.2.0", "unidic-lite>=1.0.0"]

def run(args, cwd=None):
    subprocess.run(args, cwd=cwd, check=True)

def git(path, *args):
    return subprocess.check_output(["git", *args], cwd=path, text=True).strip()

def checkout_repo(path, repo_url, revision, label, allow_local_changes=False):
    if path.is_symlink():
        raise RuntimeError(f"{label} 目录是软链接，请使用新的 runtime。")
    fresh = not path.exists()
    if fresh:
        path.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--depth", "1", "--no-checkout", repo_url + ".git", str(path)])
    if not (path / ".git").is_dir():
        raise RuntimeError(f"{label} 目录不是 Git 仓库，请使用新的 runtime。")
    if git(path, "remote", "get-url", "origin").rstrip("/").removesuffix(".git") != repo_url:
        raise RuntimeError(f"现有 {label} 不是指定仓库，请使用新的 runtime。")
    if not fresh and not allow_local_changes and git(path, "status", "--porcelain"):
        raise RuntimeError(f"现有 {label} 有未提交修改，请使用新的 runtime。")
    if fresh or git(path, "rev-parse", "HEAD") != revision:
        run(["git", "fetch", "--depth", "1", "origin", revision], cwd=path)
        run(["git", "checkout", "--force", "--detach", revision], cwd=path)

checkout_repo(COMFY_DIR, COMFY_REPO, COMFY_REVISION, "ComfyUI")
checkout_repo(NODE_DIR, NODE_REPO, NODE_REVISION, "ComfyUI-IndexTTS-2_5", allow_local_changes=True)

# 让 Linux 也走预编译的 wetext 文本规范化，避免编译 pynini
front_py = NODE_DIR / "indextts/utils/front.py"
front_py.write_text(front_py.read_text(encoding="utf-8").replace('if platform.system() != "Linux":', "if True:"), encoding="utf-8")
REQUIREMENT_FILES = [COMFY_DIR / "requirements.txt", NODE_DIR / "requirements.txt"]

skip = {"torch", "torchvision", "torchaudio", "xformers"}
lines = []
for req_file in REQUIREMENT_FILES:
    for line in req_file.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        name = re.split(r"[<>=!~\[;\s]", s, maxsplit=1)[0].strip().lower()
        if name not in skip:
            lines.append(s)
requirements = Path("/content/comfy_requirements.txt")
requirements.write_text("\n".join(lines) + "\n", encoding="utf-8")
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements), *EXTRA_PACKAGES, "huggingface_hub", "hf_xet"])

cloudflared_bin = Path("/usr/local/bin/cloudflared")
try:
    installed = subprocess.run([str(cloudflared_bin), "--version"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout
except OSError:
    installed = ""
if CLOUDFLARED_VERSION not in installed.split():
    download = cloudflared_bin.with_name("cloudflared.download")
    urllib.request.urlretrieve(
        f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64",
        str(download),
    )
    download.chmod(0o755)
    download.replace(cloudflared_bin)

print(f"ComfyUI: {COMFY_VERSION} ({COMFY_REVISION[:12]}) | ComfyUI-IndexTTS-2.5: {NODE_REVISION[:12]} | cloudflared: {CLOUDFLARED_VERSION}")
print("[OK] ComfyUI 安装完成。")

### 3. 下载模型与工作流
- **功能**：下载 IndexTTS-2.5 权重（含 QwenEmotion）与辅助编码器 / 声码器（约 5.1 GB）至 `/content/ComfyUI/models/TTS/IndexTTS-2.5`，同步工作流至 `/content/ComfyUI/user/default/workflows`。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 3. 下载模型与工作流 {display-mode: "form"}
import json, os, shutil, tempfile, time, urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import quote

os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
from huggingface_hub import hf_hub_download

COMFY_DIR = Path("/content/ComfyUI")
MODEL_DIR = COMFY_DIR / "models"
WORKFLOW_DIR = COMFY_DIR / "user/default/workflows"
WORKFLOW_URL = "https://raw.githubusercontent.com/boltguo/colab/main/workflows/indextts-2.5"
WORKFLOW_FILES = [
    "01_IndexTTS_2_5_Clone.json",
    "02_IndexTTS_2_5_Emotion_Vector.json",
    "03_IndexTTS_2_5_Emotion_Ref.json",
]
INDEXTTS = ("IndexTeam/IndexTTS-2.5", "c39ce5ba981572cb187443877ff559dfb246ce63", "TTS/IndexTTS-2.5")

MODEL_FILES = [
    (*INDEXTTS, "config.yaml", 2860),
    (*INDEXTTS, "gpt.pth", 3259599833),
    (*INDEXTTS, "s2mel.pth", 414908601),
    (*INDEXTTS, "codec.pth", 607290935),
    (*INDEXTTS, "wav2vec2bert_stats.pt", 9343),
    (*INDEXTTS, "feat1.pt", 57170),
    (*INDEXTTS, "feat2.pt", 374866),
    (*INDEXTTS, "multilingual_zh_ja_yue_char_del.tiktoken", 907395),
    (*INDEXTTS, "qwen0.6bemo4-merge/Modelfile", 360),
    (*INDEXTTS, "qwen0.6bemo4-merge/added_tokens.json", 707),
    (*INDEXTTS, "qwen0.6bemo4-merge/chat_template.jinja", 550),
    (*INDEXTTS, "qwen0.6bemo4-merge/config.json", 727),
    (*INDEXTTS, "qwen0.6bemo4-merge/generation_config.json", 117),
    (*INDEXTTS, "qwen0.6bemo4-merge/merges.txt", 1671853),
    (*INDEXTTS, "qwen0.6bemo4-merge/model.safetensors", 1192135096),
    (*INDEXTTS, "qwen0.6bemo4-merge/special_tokens_map.json", 616),
    (*INDEXTTS, "qwen0.6bemo4-merge/tokenizer.json", 11422654),
    (*INDEXTTS, "qwen0.6bemo4-merge/tokenizer_config.json", 5433),
    (*INDEXTTS, "qwen0.6bemo4-merge/vocab.json", 2776833),
]
LOADER_DIRS = {
    "UNETLoader": "diffusion_models",
    "CLIPLoader": "text_encoders",
    "VAELoader": "vae",
    "LoraLoader": "loras",
    "LoraLoaderModelOnly": "loras",
}

def local_directory(path):
    path = Path(path)
    for parent in [path, *path.parents]:
        if parent == COMFY_DIR.parent:
            break
        if parent.is_symlink():
            raise RuntimeError(f"目录是软链接，请使用新的 runtime：{parent}")
    path.mkdir(parents=True, exist_ok=True)

def model_ready(target, size):
    return target.is_file() and not target.is_symlink() and target.stat().st_size == size

def download_model(repo, revision, folder, file, size):
    base = MODEL_DIR / folder
    target = base / file
    local_directory(target.parent)
    started = time.time()
    result = Path(hf_hub_download(repo_id=repo, filename=file, revision=revision, local_dir=str(base), force_download=target.exists()))
    if result.resolve() != target.resolve() or not model_ready(target, size):
        raise RuntimeError(f"模型下载不完整：{target}")
    print(f"已下载：{file}（{size / 1024**3:.1f} GB，{time.time() - started:.0f} 秒）")

def download_models():
    pending = []
    for item in MODEL_FILES:
        target = MODEL_DIR / item[2] / item[3]
        if target.is_symlink():
            raise RuntimeError(f"模型文件是软链接：{target}")
        if model_ready(target, item[4]):
            print("已存在：", item[3])
        else:
            pending.append(item)
    if not pending:
        return
    missing = sum(item[4] for item in pending)
    free = shutil.disk_usage(MODEL_DIR).free
    if free < missing + 2 * 1024**3:
        raise RuntimeError(f"磁盘空间不足：还需 {missing / 1024**3:.1f} GB，可用 {free / 1024**3:.1f} GB。")
    started = time.time()
    pool = ThreadPoolExecutor(max_workers=3)
    try:
        futures = [pool.submit(download_model, *item) for item in sorted(pending, key=lambda item: -item[4])]
        for future in as_completed(futures):
            future.result()
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
    elapsed = time.time() - started
    print(f"模型下载完成：{missing / 1024**3:.1f} GB，用时 {elapsed:.0f} 秒（{missing / 1024**2 / max(elapsed, 1):.0f} MB/s）")

def check_workflow(data):
    if not isinstance(data, dict) or not isinstance(data.get("nodes"), list) or not data["nodes"]:
        raise ValueError("工作流 JSON 缺少 nodes。")
    for graph in [data, *data.get("definitions", {}).get("subgraphs", [])]:
        for node in graph.get("nodes", []):
            if not isinstance(node, dict) or not isinstance(node.get("type"), str):
                raise ValueError("工作流节点结构无效。")
            folder = LOADER_DIRS.get(node["type"])
            values = node.get("widgets_values")
            name = values[0] if isinstance(values, list) and values else None
            if folder and not (isinstance(name, str) and Path(name).name == name and (MODEL_DIR / folder / name).is_file()):
                raise ValueError(f"{node['type']} 引用的模型不存在：{name}")

def download_workflows():
    local_directory(WORKFLOW_DIR)
    for name in WORKFLOW_FILES:
        try:
            request = urllib.request.Request(f"{WORKFLOW_URL}/{quote(name)}", headers={"User-Agent": "colab-workflow-sync"})
            with urllib.request.urlopen(request, timeout=30) as response:
                data = json.loads(response.read())
            check_workflow(data)
            target = WORKFLOW_DIR / name
            if target.is_symlink() or (target.exists() and not target.is_file()):
                raise ValueError(f"目标是软链接或目录：{target}")
            fd, temporary = tempfile.mkstemp(prefix=".workflow-", dir=WORKFLOW_DIR)
            try:
                with os.fdopen(fd, "w", encoding="utf-8") as file:
                    json.dump(data, file, ensure_ascii=False, indent=2)
                    file.write("\n")
                os.replace(temporary, target)
            finally:
                if os.path.exists(temporary):
                    os.unlink(temporary)
        except Exception as error:
            raise RuntimeError(f"工作流同步失败：{name}：{error}") from error
        print("工作流：", name)

for path in [MODEL_DIR, COMFY_DIR / "input", COMFY_DIR / "output"]:
    local_directory(path)
download_models()
import sys
NODE_DIR = COMFY_DIR / "custom_nodes/ComfyUI-IndexTTS-2_5"
if str(NODE_DIR) not in sys.path:
    sys.path.insert(0, str(NODE_DIR))
from indextts.utils.model_download import ensure_models_available
ensure_models_available(str(MODEL_DIR / "TTS/IndexTTS-2.5"))
sample_audio = NODE_DIR / "example/ref_audio.mp3"
if sample_audio.is_file() and not (COMFY_DIR / "input/ref_audio.mp3").exists():
    shutil.copy2(sample_audio, COMFY_DIR / "input/ref_audio.mp3")
    print("示例音频：", COMFY_DIR / "input/ref_audio.mp3")
download_workflows()
print("[OK] 模型与工作流准备完成。")

### 4. 启动并打开 ComfyUI
- **功能**：后台启动 ComfyUI（`127.0.0.1:8188`），启动 Cloudflare Tunnel 并检查公网访问。
- **隧道模式**：默认不勾选 `USE_CF_TUNNEL_TOKEN`，直接启动 Cloudflare Quick Tunnel 临时公网域名（用完即关）；勾选后从 Colab 左侧 **🔑 Secrets** 读取 `CF_TUNNEL_TOKEN` 挂载自有域名。
- **使用**：直接运行，点击输出的公网链接访问。

In [ ]:
# @title 4. 启动并打开 ComfyUI {display-mode: "form"}
USE_CF_TUNNEL_TOKEN = False  # @param {type:"boolean"}
import json, os, re, socket, subprocess, sys, time, urllib.request
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
PORT = 8188
BASE = f"http://127.0.0.1:{PORT}"
LOG = Path("/content/comfyui.log")
LAUNCHER = Path("/content/comfy_userdata_compat.py")
LAUNCH_CONFIG = Path("/content/comfyui.launch.json")
PROCESS_PATTERN = r"comfy_userdata_compat\.py|ComfyUI/main\.py"
LAUNCH_ARGS = [
    "--output-directory", str(COMFY_DIR / "output"),
    "--enable-cors-header",
    "--enable-compress-response-body",
    "--disable-all-custom-nodes",
    "--whitelist-custom-nodes", "ComfyUI-IndexTTS-2_5",
    "--disable-partner-nodes",
]
REQUIRED_NODES = {"IndexTTS2_5Node", "EmoSettingsNode", "LoadAudio", "SaveAudioMP3"}

CF_BIN = Path("/usr/local/bin/cloudflared")
CF_LOG = Path("/content/cloudflared.log")

def read_text(path):
    return path.read_text(errors="replace") if path.exists() else ""

def tail(path, size=8000):
    return read_text(path)[-size:]

def rotate_log(path):
    if path.is_file() and path.stat().st_size:
        path.replace(path.with_name(path.name + ".1"))

def process_running(pattern):
    return subprocess.run(["pgrep", "-f", pattern], stdout=subprocess.DEVNULL).returncode == 0

def stop_processes(pattern, timeout):
    subprocess.run(["pkill", "-f", pattern], check=False)
    deadline = time.time() + timeout
    while process_running(pattern) and time.time() < deadline:
        time.sleep(2)
    if process_running(pattern):
        subprocess.run(["pkill", "-9", "-f", pattern], check=False)
        time.sleep(2)

def resolve_cf_token():
    t = ""
    try:
        from google.colab import userdata
        t = (userdata.get("CF_TUNNEL_TOKEN") or "").strip()
    except Exception:
        t = ""
    if not t:
        t = os.environ.get("CF_TUNNEL_TOKEN", "").strip()
    return t.split()[-1] if t else ""

def start_cf_tunnel(token=""):
    if not CF_BIN.exists():
        raise RuntimeError("未找到 cloudflared，请先运行第 2 步。")
    stop_processes("cloudflared tunnel", 10)
    rotate_log(CF_LOG)
    cmd = [
        str(CF_BIN),
        "tunnel",
        "--no-autoupdate",
        "--protocol", "http2",
        "--edge-ip-version", "4",
    ]
    if token:
        cmd.extend(["run", "--url", BASE, "--token", token])
    else:
        cmd.extend(["--url", BASE])
    with CF_LOG.open("w") as cf_log:
        cf_proc = subprocess.Popen(
            cmd,
            stdin=subprocess.DEVNULL,
            stdout=cf_log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    deadline = time.time() + (30 if token else 60)
    while time.time() < deadline:
        if cf_proc.poll() is not None:
            print(tail(CF_LOG, 4000))
            if token:
                raise RuntimeError("Cloudflare Tunnel 启动失败，请检查 CF_TUNNEL_TOKEN 是否正确。")
            raise RuntimeError("Cloudflare 临时隧道启动失败，请查看上方日志。")
        log_text = read_text(CF_LOG)
        if not token:
            match = re.search(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
            if match and "Registered tunnel connection" in log_text:
                url = match.group(0)
                print(f"[OK] Cloudflare 临时隧道已连接 (HTTP/2): {url} -> {BASE}")
                return url
        elif "Registered tunnel connection" in log_text:
            if "Updated to new configuration" not in log_text:
                time.sleep(1)
                log_text = read_text(CF_LOG)
            configs = [line for line in log_text.splitlines() if "Updated to new configuration" in line]
            config = configs[-1].replace('\\"', '"') if configs else ""
            routes = re.findall(r'"hostname"\s*:\s*"([^"]+)"(?:(?!"hostname").)*?"service"\s*:\s*"([^"]+)"', config)
            for host, service in routes:
                print(f"[OK] Cloudflare Tunnel 已连接 (HTTP/2): https://{host} -> {service}")
            hosts = [host for host, service in routes if f":{PORT}" in service]
            if not routes:
                print("[OK] Cloudflare Tunnel 已连接 (HTTP/2)，请通过你在 Cloudflare 绑定的域名访问。")
            elif not hosts:
                print(f"[WARN] 没有域名转发到本服务的 {PORT} 端口！")
                print(f"       请前往 Cloudflare Zero Trust -> Networks -> Tunnels -> Public Hostname，将 Service URL 改为 {BASE}")
            return f"https://{hosts[0]}" if hosts else "https://<你的CF域名>"
        time.sleep(1)
    if not token:
        match = re.search(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com", read_text(CF_LOG))
        if match:
            url = match.group(0)
            print(f"[OK] Cloudflare 临时隧道已创建: {url} -> {BASE}")
            return url
        print(tail(CF_LOG, 4000))
        raise RuntimeError(f"Cloudflare 临时隧道 60 秒内未返回地址，详见 {CF_LOG}。")
    print(f"[WARN] Cloudflare Tunnel 30 秒内未确认连接，仍在后台运行，详见 {CF_LOG}。")
    return "https://<你的CF域名>"

def check_public_url(url, route):
    if "<" in url:
        return
    error = None
    deadline = time.time() + 60
    while time.time() < deadline:
        try:
            request = urllib.request.Request(url + route, headers={"User-Agent": "colab-tunnel-check"})
            with urllib.request.urlopen(request, timeout=10):
                print(f"[OK] 公网访问正常：{url}")
                return
        except Exception as e:
            error = e
        time.sleep(3)
    print(f"[WARN] 公网地址 60 秒内无法访问：{url}{route}（{error}）")

cf_token = resolve_cf_token() if USE_CF_TUNNEL_TOKEN else ""
if USE_CF_TUNNEL_TOKEN and not cf_token:
    raise RuntimeError("已勾选 USE_CF_TUNNEL_TOKEN，但未检测到 CF_TUNNEL_TOKEN；请在 Colab 左侧 🔑 Secrets 中添加 CF_TUNNEL_TOKEN 并开启访问权限，或取消勾选改用临时隧道。")

if not (COMFY_DIR / "main.py").is_file():
    raise RuntimeError("未找到 ComfyUI，请先运行第 2 步。")

LAUNCHER_SOURCE = r'''
"""Start pinned ComfyUI with narrow workflow aliases and Cloudflare compatibility."""
import os
import runpy
import sys
from pathlib import Path
from aiohttp import web


def add_workflow_aliases(manager, routes):
    originals = [route for route in routes
                 if getattr(route, "path", None) == "/userdata/{file}"
                 and getattr(route, "method", None) in {"GET", "POST", "DELETE"}]
    if {route.method for route in originals} != {"GET", "POST", "DELETE"}:
        raise RuntimeError("ComfyUI 工作流访问接口已变化，未安装兼容路由。")
    for route in originals:
        async def workflow_alias(request, handler=route.handler):
            file = request.match_info.get("file", "")
            user_root = manager.get_request_user_filepath(request, None, create_dir=False)
            candidate = manager.get_request_user_filepath(request, file, create_dir=False)
            if not user_root or not candidate:
                return web.Response(status=403)
            workflow_directory = os.path.join(user_root, "workflows")
            if os.path.islink(workflow_directory):
                return web.Response(status=403)
            workflow_root = os.path.realpath(workflow_directory)
            try:
                within = os.path.commonpath((workflow_root, os.path.realpath(candidate))) == workflow_root
            except ValueError:
                within = False
            if not within or os.path.realpath(candidate) == workflow_root:
                return web.Response(status=403)
            return await handler(request)
        routes.route(route.method, "/userdata/{file:workflows/[^/]+}")(workflow_alias)


def install_cf_middleware():
    import middleware.cache_middleware as cm
    orig_cache_control = cm.cache_control

    @web.middleware
    async def cf_safe_middleware(request, handler):
        response = await orig_cache_control(request, handler)
        if request.method == "GET" and request.path == "/" and getattr(response, "status", None) == 200:
            import server
            inst = getattr(server.PromptServer, "instance", None)
            web_root = getattr(inst, "web_root", None) if inst else None
            index_file = Path(web_root) / "index.html" if web_root else None
            if index_file and index_file.is_file():
                html = index_file.read_text(encoding="utf-8", errors="replace")
                html = html.replace("<script ", '<script data-cfasync="false" ')
                new_resp = web.Response(text=html, content_type="text/html")
                for k, v in response.headers.items():
                    if k.lower() not in {"content-length", "content-type", "etag", "last-modified"}:
                        new_resp.headers[k] = v
                new_resp.headers["Cache-Control"] = "no-store, no-transform, must-revalidate"
                response = new_resp
        if isinstance(response, web.StreamResponse) and not isinstance(response, web.WebSocketResponse) and not response.prepared:
            if getattr(response, "status", None) == 200 and (
                request.path.startswith("/assets/")
                or request.path.endswith((".woff2", ".woff", ".ttf"))
            ):
                response.headers["Cache-Control"] = "public, max-age=604800, immutable, no-transform"
            else:
                cc = response.headers.get("Cache-Control", "")
                if "no-transform" not in cc:
                    response.headers["Cache-Control"] = f"{cc}, no-transform" if cc else "no-transform"
            origin = request.headers.get("Origin")
            if origin and response.headers.get("Access-Control-Allow-Origin") == "*":
                response.headers["Access-Control-Allow-Origin"] = origin
                response.headers["Vary"] = "Origin"
        return response

    cm.cache_control = cf_safe_middleware


def main():
    comfy_root = Path("/content/ComfyUI")
    if comfy_root.is_symlink() or not (comfy_root / "main.py").is_file() or not (comfy_root / "comfy/options.py").is_file():
        raise RuntimeError("ComfyUI 源码目录缺失或无效。")
    os.chdir(comfy_root)
    sys.path.insert(0, str(comfy_root))
    sys.argv[0] = str(comfy_root / "main.py")
    import comfy.options
    comfy.options.enable_args_parsing()
    install_cf_middleware()
    from app.user_manager import UserManager
    original = UserManager.add_routes

    def compatible_routes(self, routes):
        original(self, routes)
        add_workflow_aliases(self, routes)

    UserManager.add_routes = compatible_routes
    runpy.run_path("main.py", run_name="__main__")


if __name__ == "__main__":
    main()
'''

def get_json(route, timeout=5):
    with urllib.request.urlopen(BASE + route, timeout=timeout) as response:
        return json.load(response)

def ready(timeout=5):
    try:
        get_json("/system_stats", timeout)
        return True
    except Exception:
        return False

def get_listen_addrs():
    try:
        with socket.socket(socket.AF_INET6, socket.SOCK_STREAM) as s:
            s.bind(("::1", 0))
        return "127.0.0.1,::1"
    except OSError:
        return "127.0.0.1"

launch_config = json.dumps({"launcher": LAUNCHER_SOURCE, "args": LAUNCH_ARGS})
running = ready() or (process_running(PROCESS_PATTERN) and ready(timeout=30))
if running and read_text(LAUNCH_CONFIG) != launch_config:
    try:
        queue = get_json("/queue", timeout=10)
        busy = bool(queue["queue_running"] or queue["queue_pending"])
    except Exception:
        busy = True
    if busy:
        print("[WARN] 启动配置已更新，但 ComfyUI 正在执行任务；任务完成后重新运行本步骤即可应用。")
    else:
        print("启动配置已更新，正在重启 ComfyUI...")
        stop_processes(PROCESS_PATTERN, 60)
        running = False

if running:
    print("[OK] ComfyUI 已经在运行。")
else:
    stop_processes(PROCESS_PATTERN, 60)
    LAUNCHER.write_text(LAUNCHER_SOURCE)
    rotate_log(LOG)
    with LOG.open("w") as log:
        process = subprocess.Popen(
            [sys.executable, str(LAUNCHER), "--listen", get_listen_addrs(), "--port", str(PORT), *LAUNCH_ARGS],
            cwd=COMFY_DIR,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    LAUNCH_CONFIG.write_text(launch_config)
    print("正在启动 ComfyUI...")
    deadline = time.time() + 600
    while time.time() < deadline:
        if process.poll() is not None:
            print(tail(LOG))
            raise RuntimeError(f"ComfyUI 启动失败（退出码 {process.returncode}），请查看上方日志。")
        if ready():
            print("[OK] ComfyUI 启动成功。")
            break
        time.sleep(2)
    else:
        print(tail(LOG))
        raise TimeoutError("ComfyUI 10 分钟内未就绪，请查看上方日志。")

missing = sorted(REQUIRED_NODES - set(get_json("/object_info", timeout=60)))
if missing:
    print(tail(LOG))
    raise RuntimeError(f"缺少节点：{', '.join(missing)}。请重新运行第 2 步安装指定版本后，再运行本步骤。日志：{LOG}")

host_url = start_cf_tunnel(cf_token)
check_public_url(host_url, "/system_stats")